## Вопрос

Какие короткие правила выбора хода сохраняют результат на маленькой доске? Какие из них помогают и после усложнения игры?

В [главе о крестиках-ноликах](tic-tac-toe.ipynb) мы разделили два вопроса. Какой исход можно гарантировать из позиции и какой исход гарантирует ход, выбранный нашим правилом?

## Этапы исследования

| Этап | Что фиксируем | Какая проверка |
|---|---|---|
| Маленькая доска | Мало фигур, начальная позиция, правила хода и взятия | Полный граф достижимых состояний |
| Следующий размер | Меняем один параметр за раз, когда это возможно | Повторяем проверку тех же правил и ищем различия |
| Доска 8×8 | Конкретный вариант шашек и набор позиций | Для выбранных эндшпилей — точный анализ, где он выполним. Для остальных — эксперимент с указанными границами |

Для каждой модели запишем обязательность взятия, цепочки взятий, превращение в дамку, критерий проигрыша и правила ничьей. Если решение о ничьей зависит от повторов, при построении графа учтём нужную историю. Одинаковая расстановка может иметь разные продолжения в зависимости от этой истории.

## Из каких частей состоит алгоритм

1. **Представление состояния.** Расстановка, сторона хода и сведения, необходимые для применения правил.
2. **Допустимые ходы.** Получить все законные ходы, затем построить состояние после каждого.
3. **Эталон.** На маленькой модели найти исход при оптимальной игре. Для графа с циклами использовать анализ из конечных состояний с явно заданным правилом ничьей.
4. **Короткое правило.** Выбрать среди допустимых ходов. Например, сначала сравнить число взятых фигур, а равенство разрешить в заданном порядке.
5. **Проверка.** На одной и той же позиции сравнить эталонную гарантию с гарантией при применении правила. После этого отдельно посчитать результаты в партиях против выбранных соперников.

## Как сравнить результаты

В коде ниже используются четыре условные позиции. Для каждой сравниваем исход при оптимальной игре с исходом, который гарантирует простое правило против любого допустимого ответа. Шкала та же — победа +1, ничья 0, поражение −1.

In [1]:
def compare_guarantees(optimal, rule):
    if optimal.keys() != rule.keys():
        raise ValueError("Нужен один и тот же набор позиций")

    same = [position for position in optimal if rule[position] == optimal[position]]
    missed_wins = [
        position for position in optimal
        if optimal[position] == 1 and rule[position] < 1
    ]
    avoidable_losses = [
        position for position in optimal
        if optimal[position] >= 0 and rule[position] == -1
    ]
    return same, missed_wins, avoidable_losses


optimal = {"A": 1, "B": 1, "C": 0, "D": -1}
simple_rule = {"A": 1, "B": 0, "C": -1, "D": -1}

same, missed_wins, avoidable_losses = compare_guarantees(optimal, simple_rule)
print("Та же гарантия:", ", ".join(same))
print("Упущенный выигрыш:", ", ".join(missed_wins))
print("Предотвратимое поражение:", ", ".join(avoidable_losses))

Та же гарантия: A, D
Упущенный выигрыш: B
Предотвратимое поражение: C


Категории «упущенная победа» и «предотвратимое поражение» могут пересекаться. Например, если правило проигрывает из выигрышной позиции, она войдёт в обе категории.

## Контрпримеры

Для каждого найденного случая сохраним позицию, чей ход, точную формулировку правила, выбранный им ход и ответ соперника. Если правило смотрит на несколько ходов вперёд, запишем последовательность, достаточную для проверки результата.

## Первый практический шаг

Выбрать одну упрощённую модель, описать её правила и вручную проверить допустимые ходы в нескольких позициях. Эти примеры послужат проверкой алгоритма перед полным перебором.

[Выбрать следующую игру или отдельный контрпример](../next-games.qmd)